# Notebook 01: Setup & Data Overview

**Phase 0 — Scaffolding & Data Pipeline**

This notebook validates the data pipeline end-to-end and serves as a reference
for the structure of the TPC-H dataset used throughout this project.

---

## Prerequisites

Before running this notebook:

```bash
# 1. Copy the env template and fill in credentials
cp .env.example .env

# 2. Start PostgreSQL
docker compose up -d

# 3. Generate Parquet files (~30–60 s)
uv run python scripts/generate_data.py

# 4. Seed PostgreSQL (~1–3 min)
uv run python scripts/seed_postgres.py
```

In [2]:
import pathlib
import sys

import duckdb
import pandas as pd

# Ensure the project root is on sys.path so ``config`` can be imported
# regardless of where JupyterLab was launched from.
ROOT = pathlib.Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PARQUET_DIR = ROOT / "data" / "parquet"

# DuckDB in-memory connection for querying Parquet files directly.
duck = duckdb.connect()

print(f"DuckDB version : {duckdb.__version__}")
print(f"Parquet dir    : {PARQUET_DIR}")
print(f"Parquet exists : {PARQUET_DIR.exists()}")

DuckDB version : 1.5.0
Parquet dir    : B:\Projects\sql-and-duckdb-playbook\notebooks\data\parquet
Parquet exists : False


---

## 1 · Why TPC-H?

TPC-H is the **industry-standard OLAP benchmark** published by the Transaction
Processing Performance Council.  It is chosen here for three reasons:

| Reason | Detail |
|:---|:---|
| **Reproducible** | Generated entirely in-process by DuckDB's `dbgen` extension — no download, no external dependency |
| **Principled** | Queries in the benchmarking phase (notebook 07) are anchored to the official TPC-H Q1/Q3/Q5 spec — not cherry-picked |
| **Realistic shape** | 8 normalised tables, 6 M+ rows in `lineitem`, real OLAP query patterns (aggregations, multi-table joins, date range filters) |

Scale factor 1 (`sf=1`) produces approximately **1 GB** of data — large enough
to make query plan differences observable, small enough to fit on a laptop.

---

## 2 · Schema Overview

```
  region (5)
    └─ nation (25)
         ├─ supplier (10 K)
         │     └─ partsupp (800 K) ──┐
         │                           │
         │    part (200 K) ──────────┤
         │                           │
         └─ customer (150 K)         │
               └─ orders (1.5 M)     │
                     └─ lineitem (6 M, references partsupp)
```

Row counts above are approximate for `sf=1`.  `lineitem` is the fact table —
almost every analytical query in this repo touches it.

---

## 3 · Row Counts

In [ ]:
tables = [
    "region", "nation", "supplier", "part",
    "partsupp", "customer", "orders", "lineitem",
]

rows = []
for table in tables:
    parquet = PARQUET_DIR / f"{table}.parquet"
    count = duck.execute(
        f"SELECT COUNT(*) FROM read_parquet('{parquet}')"
    ).fetchone()[0]
    size_mb = parquet.stat().st_size / 1_048_576
    rows.append({"table": table, "rows": count, "parquet_size_mb": round(size_mb, 1)})

df_counts = pd.DataFrame(rows)
df_counts["rows"] = df_counts["rows"].map("{:,}".format)
df_counts.columns = ["Table", "Row Count", "Parquet Size (MB)"]
df_counts

---

## 4 · Table Summaries via DuckDB `SUMMARIZE`

`SUMMARIZE` is a DuckDB-exclusive shortcut that computes per-column statistics
(count, nulls, min, max, mean, std) in a single pass — equivalent to running
`DESCRIBE` + multiple `SELECT` aggregates, but in one line.  It is one of the
DuckDB-specific features demonstrated in notebook 05.

In [ ]:
# SUMMARIZE lineitem — the most important table in TPC-H.
lineitem_parquet = PARQUET_DIR / "lineitem.parquet"
duck.execute(f"SUMMARIZE SELECT * FROM read_parquet('{lineitem_parquet}')").df()

In [ ]:
# SUMMARIZE orders — useful to see the date range and price distribution.
orders_parquet = PARQUET_DIR / "orders.parquet"
duck.execute(f"SUMMARIZE SELECT * FROM read_parquet('{orders_parquet}')").df()

---

## 5 · Sample Data

In [ ]:
# Five rows from lineitem — shows the granularity: one row per order line.
duck.execute(
    f"SELECT * FROM read_parquet('{lineitem_parquet}') LIMIT 5"
).df()

In [ ]:
# A denormalised view: lineitem joined to orders and customer.
# This is the shape most window function queries operate on.
duck.execute(f"""
    SELECT
        o.o_orderkey,
        o.o_orderdate,
        c.c_name        AS customer,
        o.o_orderstatus AS status,
        l.l_linenumber  AS line,
        l.l_extendedprice * (1 - l.l_discount) AS net_price
    FROM read_parquet('{PARQUET_DIR}/lineitem.parquet')  l
    JOIN read_parquet('{PARQUET_DIR}/orders.parquet')    o
        ON l.l_orderkey = o.o_orderkey
    JOIN read_parquet('{PARQUET_DIR}/customer.parquet')  c
        ON o.o_custkey = c.c_custkey
    LIMIT 10
""").df()

---

## 6 · PostgreSQL Connectivity Check

In [ ]:
import psycopg2
from config import settings

pg = psycopg2.connect(settings.dsn)

with pg.cursor() as cur:
    cur.execute("""
        SELECT
            schemaname,
            tablename,
            pg_size_pretty(pg_total_relation_size(schemaname || '.' || tablename)) AS size
        FROM pg_tables
        WHERE schemaname = 'public'
        ORDER BY pg_total_relation_size(schemaname || '.' || tablename) DESC
    """)
    pg_tables = cur.fetchall()

pg.close()

pd.DataFrame(pg_tables, columns=["Schema", "Table", "Total Size"])

---

## Summary

| ✅ | Item |
|:---|:---|
| ✅ | DuckDB `dbgen(sf=1)` generated 8 TPC-H tables |
| ✅ | All tables exported to `data/parquet/` (flat + year-partitioned orders) |
| ✅ | `sql/schema/tpch_tables.sql` DDL created all 8 PostgreSQL tables |
| ✅ | `seed_postgres.py` bulk-loaded all rows via psycopg2 `COPY FROM STDIN` |
| ✅ | PostgreSQL row counts and table sizes confirmed above |

**Next:** [Notebook 02 — Window Functions](02_window_functions.ipynb)